In [37]:
import pandas as pd
import plotly.graph_objects  as go
import numpy as np
import plotly.express as px

In [2]:
import mlflow
print(mlflow.__version__)

3.3.2


In [3]:
url = "https://drive.google.com/uc?export=download&id=1YkRYSkAHeH7nMgO-ums8zbU7paODGs3t"

In [4]:
df = pd.read_csv(url)

In [5]:
print(df.info())
print(df.head(3))
print(df["length_of_stay"].value_counts())
print(df["length_of_stay"].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patient_id                 8000 non-null   object 
 1   admission_date             8000 non-null   object 
 2   season                     8000 non-null   object 
 3   age                        8000 non-null   int64  
 4   gender                     8000 non-null   object 
 5   region                     8000 non-null   object 
 6   primary_diagnosis          8000 non-null   object 
 7   comorbidities_count        8000 non-null   int64  
 8   length_of_stay             8000 non-null   int64  
 9   treatment_type             8000 non-null   object 
 10  medications_count          8000 non-null   int64  
 11  followup_visits_last_year  8000 non-null   int64  
 12  prev_readmissions          8000 non-null   int64  
 13  insurance_type             8000 non-null   objec

In [6]:
# liste de colonnes source de data leakage pour la prediction de length_of_stay.
# Hypothèse de travail :  la prédiction intervient après le premier diagnositic les données médicales sont donc connues.

target = "length_of_stay"
columns_to_exclude = ["discharge_disposition","readmission_risk_score","label"]

df_ml = df.drop(columns= columns_to_exclude)

#Selection des colonnes numériques
num_col = df_ml.select_dtypes(include="number").columns.drop(target)

#Vérification de la corrélation des colonnes numériques avec la target.

correlation = df_ml[num_col].corrwith(df_ml[target]).sort_values()
print(correlation)


prev_readmissions            0.146806
followup_visits_last_year    0.163643
age                          0.205709
medications_count            0.213912
comorbidities_count          0.243239
dtype: float64


On peut noter que toutes les corrélations sont positives. Ce qui est logique d'un point de vue médicale. Cependant elles sont aussi toutes inférieures à 0.30 ce qui souligne une relation plutôt faible entre les colonnes numériques est la target [durée de séjour]
Il reste à vérifier la corrélations entre la target et les colonnes catégorielles

In [7]:
columns_cat_to_exclude = ["patient_id","admission_date"]
cat_col = df_ml.select_dtypes(include="object").columns.drop(columns_cat_to_exclude)

Pour éviter le biais d'ordre impliqué par le labelEncoding est colonnes catégorielles je vais utiliser le ratio de corrélation (Eta carré) 

In [8]:
def calculate_eta_squared(data, cat_cols, target_col):
    # Moyenne globale de la target
    grand_mean = data[target_col].mean()
    
    # Somme des carrés totale (SST => Sum of Squares Total)
    sst = ((data[target_col] - grand_mean) ** 2).sum()
    
    # Somme des carrés inter-groupes (SSB => Sum of Squares Between)
    # On regarde l'écart entre la moyenne de chaque catégorie et la moyenne globale
    category_means = data.groupby(cat_cols)[target_col].mean()
    category_counts = data.groupby(cat_cols)[target_col].count()
    
    ssb = (category_counts * (category_means - grand_mean) ** 2).sum()
    
    return ssb / sst if sst != 0 else 0

In [9]:
corr_score ={}

for col in cat_col :
    score = calculate_eta_squared(df_ml,col,target)
    corr_score[col] = score

correlation_resultat = pd.Series(corr_score).sort_values(ascending=False)

print(correlation_resultat)

primary_diagnosis    0.653437
season               0.078522
insurance_type       0.021054
treatment_type       0.000964
region               0.000308
gender               0.000103
dtype: float64


Il existe un très forte corrélation entre primary_diagnosis et la target 65%. (Limite du dataset synthétique ?) risque la régression linaire risque de prédire la moyenne par primary_diagosis. Vérifier la distribution par primary_diagnisis. Si les moyennes sont similaires la régréssion linéaire risque d'être limitée.

In [10]:
diag_mean_analysis = (df_ml.groupby("primary_diagnosis")["length_of_stay"].
                      agg(["mean","std","count"]).
                      sort_values("mean"))
diag_mean_analysis = diag_mean_analysis.round(2)
print(diag_mean_analysis)

                    mean   std  count
primary_diagnosis                    
Hypertension        5.82  1.14   1301
Appendicitis        5.88  1.14    534
Diabetes            6.75  1.14   1470
Influenza           6.79  1.13    233
Kidney Disease      7.74  1.12    868
Fracture            7.81  1.14    434
Pneumonia           8.70  1.06    425
COPD                8.83  1.16    952
Stroke              9.74  1.17    883
Heart Failure       9.75  1.10    403
Sepsis             10.76  1.14    497


Les moyennes sont relativement distinctes par diagnosis et on distingue des groupes. les écart-type sont homogènes et limités ce qui peut être une explication de l'eta-squared à 65%. les groupes sont serrés et bien séparés.
Le risque d'une réstitution de la moyenne par le modèle reste présent. cependant la prédiction peut être ajuster par les colonnes numérique à la marge.

In [11]:
fig = px.histogram(df, x="primary_diagnosis", 
                   title="Distribution des Diagnostics Principaux",
                   labels={'primary_diagnosis': 'Diagnostic', 'count': 'Nombre de Patients'},
                   category_orders={"primary_diagnosis": df["primary_diagnosis"].value_counts().index}) # Tri par fréquence

# Amélioration de la lisibilité si les noms sont longs
fig.update_layout(xaxis_tickangle=-45) 
fig.show()

In [51]:
# Stack ml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,root_mean_squared_error
from sklearn.base import BaseEstimator, TransformerMixin

from xgboost import XGBRegressor
import xgboost as xgb


import sklearn
print(f"sklearn version {sklearn.__version__}")

import mlflow
import mlflow.sklearn
print(f"mlflow version {mlflow.__version__}")
print(f"xgboost version {xgb.__version__}")


sklearn version 1.7.1
mlflow version 3.3.2
xgboost version 3.2.0


In [13]:
# Identification des feature et e la target

target = "length_of_stay"
columns_leakage = ["discharge_disposition","readmission_risk_score","label"]
columns_cat_useless = ["patient_id","admission_date"]
col_to_drop = columns_leakage + columns_cat_useless

df_ml = df.drop(columns = col_to_drop)


X = df_ml.drop(columns = target)
y = df_ml[target]

num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = X.select_dtypes(include="object").columns.tolist()

print("Numériques :", num_features)
print("Catégorielles :", cat_features)
print("Shape X :", X.shape)


Numériques : ['age', 'comorbidities_count', 'medications_count', 'followup_visits_last_year', 'prev_readmissions']
Catégorielles : ['season', 'gender', 'region', 'primary_diagnosis', 'treatment_type', 'insurance_type']
Shape X : (8000, 11)


In [14]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")


Train : (6400, 11) | Test : (1600, 11)


In [ ]:
# Pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Sous-pipeline catégoriel
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Assemblage
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])


In [ ]:
# Pipeline linearRegession
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [33]:
# Pipeline RandomForest
model_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",    RandomForestRegressor(
                        n_estimators=100,
                        random_state=0,
                        n_jobs=-1
                    ))
])

In [46]:
# Pipeline Xgboost

model_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",    XGBRegressor(
                        n_estimators=100,
                        learning_rate=0.1,
                        max_depth=4,
                        random_state=42,
                        n_jobs=-1
                    ))
])


In [ ]:
#Regression  LinearRegression

mlflow.set_experiment("length_of_stay_prediction")
cv = KFold(n_splits=5, shuffle=True, random_state=0)

In [63]:
with mlflow.start_run(run_name="linear_regression_baseline"):

    # --- Cross-validation sur le train set ---
    cv_results = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"],
        return_train_score=True
    )

    cv_mae  = -cv_results["test_neg_mean_absolute_error"].mean()
    cv_rmse = -cv_results["test_neg_root_mean_squared_error"].mean()
    cv_r2   =  cv_results["test_r2"].mean()

    print("=== Cross-validation (train set) ===")
    print(f"MAE  : {cv_mae:.3f} jours")
    print(f"RMSE : {cv_rmse:.3f} jours")
    print(f"R²   : {cv_r2:.3f}")

    # --- Entraînement final ---
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    test_mae  = mean_absolute_error(y_test, y_pred)
    test_r2   = r2_score(y_test, y_pred)
    test_rmse = root_mean_squared_error(y_test, y_pred)

    print("\n=== Test set ===")
    print(f"MAE  : {test_mae:.3f} jours")
    print(f"RMSE : {test_rmse:.3f} jours")
    print(f"R²   : {test_r2:.3f}")

    # --- Logging mlflow ---
    mlflow.log_params({
        "model": "LinearRegression",
        "test_size": 0.2,
        "cv_folds": 5,
        "num_features": num_features,
        "cat_features": cat_features
    })

    mlflow.log_metrics({
        "cv_mae": cv_mae,
        "cv_rmse": cv_rmse,
        "cv_r2": cv_r2,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

    mlflow.sklearn.log_model(model, "linear_regression_baseline")

    print("\n✓ Run loggé dans mlflow")

2026/03/17 20:53:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


=== Cross-validation (train set) ===
MAE  : 0.823 jours
RMSE : 1.033 jours
R²   : 0.710

=== Test set ===
MAE  : 0.803 jours
RMSE : 1.013 jours
R²   : 0.736


2026/03/17 20:53:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



✓ Run loggé dans mlflow


En l'etat le modèle semble avoir de bon résultat au globale avec un R2 score à 0.736/ cependant il faut vérfier au niveau du primary_diagnosis si c'est toujours le cas

In [18]:
# 1. Créer un DataFrame de résultats sur le set de Test
results = X_test.copy()
results["actual"] = y_test
results["predicted"] = model.predict(X_test)

# 2. Fonction pour calculer le R2
def get_r2(group):
    # Le R2 nécessite au moins 2 points pour être calculé
    if len(group) < 2:
        return None
    return r2_score(group["actual"], group["predicted"])

# 3. GroupBy et calcul
r2_per_diagnosis = results.groupby("primary_diagnosis").apply(get_r2)

print(r2_per_diagnosis.sort_values())

primary_diagnosis
Sepsis            0.039775
Hypertension      0.124376
Appendicitis      0.130416
Pneumonia         0.130797
Diabetes          0.165278
Heart Failure     0.168464
Stroke            0.210955
COPD              0.218623
Kidney Disease    0.247797
Influenza         0.255039
Fracture          0.255611
dtype: float64


C:\Users\muren\AppData\Local\Temp\ipykernel_42768\2161151008.py:14: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



En effet le r2 score par primary_diagnosis n'est pas bon le modèle fait à peine mieux que la moyenne et dans certain cas comme le Sepsis ou Hypertension il prédit la moyenne.

In [29]:
# Calcul de la moyenne par diagnostic sur le train (pour éviter le leakage)
baseline_means = df.groupby("primary_diagnosis")[target].mean()

# On mappe cette moyenne sur le set de test
results["baseline_pred"] = results["primary_diagnosis"].map(baseline_means)

# Comparaison des erreurs
results["error_model"] = (results["actual"] - results["predicted"]).abs()
results["error_baseline"] = (results["actual"] - results["baseline_pred"]).abs()


df_results = results.groupby("primary_diagnosis")[["error_model","error_baseline","baseline_pred"]].mean()

print(df_results)

                   error_model  error_baseline  baseline_pred
primary_diagnosis                                            
Appendicitis          0.795835        0.816059       5.878277
COPD                  0.747986        0.846261       8.826681
Diabetes              0.836069        0.890076       6.753741
Fracture              0.680499        0.778724       7.813364
Heart Failure         0.752384        0.844138       9.746898
Hypertension          0.847333        0.894626       5.820907
Influenza             0.733540        0.871858       6.793991
Kidney Disease        0.714972        0.823178       7.741935
Pneumonia             0.828675        0.928816       8.701176
Sepsis                0.885955        0.966783      10.758551
Stroke                0.884975        1.022266       9.742922


Le gain sur le modèle naïf est réel mais modeste — entre 0.05 et 0.14 jour de gain selon le diagnostic. C'est la limite naturelle de la régression linéaire : elle apprend bien les moyennes par groupe mais ne capte pas les interactions entre features. Un modèle non linéaire devrait creuser cet écart, particulièrement sur Sepsis et Stroke où la complexité clinique est la plus forte.

In [30]:
from plotly.subplots import make_subplots

df_results = df_results.sort_values("baseline_pred")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("MAE : modèle vs naïf par diagnostic",
                    "Durée moyenne vs erreur modèle")
)

# Viz 1 — Barres groupées
fig.add_trace(go.Bar(
    name="Modèle LR",
    x=df_results.index,
    y=df_results["error_model"],
    marker_color="#4C78A8"
), row=1, col=1)

fig.add_trace(go.Bar(
    name="Modèle naïf",
    x= df_results.index,
    y= df_results["error_baseline"],
    marker_color="#F58518"
), row=1, col=1)

# Viz 2 — Scatter durée vs erreur
fig.add_trace(go.Scatter(
    x=df_results["baseline_pred"],
    y= df_results["error_model"],
    mode="markers+text",
    text= df_results.index,
    textposition="top center",
    marker=dict(size=10, color="#4C78A8"),
    showlegend=False
), row=1, col=2)

fig.update_layout(
    barmode="group",
    template="plotly_white",
    title="Analyse des erreurs par diagnostic",
    height=500
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_yaxes(title_text="MAE (jours)", row=1, col=1)
fig.update_xaxes(title_text="Durée moyenne séjour (jours)", row=1, col=2)
fig.update_yaxes(title_text="MAE modèle (jours)", row=1, col=2)

fig.show()

Viz 1 : 

La barre orange (naïf) est systématiquement au dessus de la barre bleue (modèle) — le modèle bat le naïf sur tous les diagnostics. L'écart est visuellement homogène, entre 0.05 et 0.15 jour selon le diagnostic. Stroke est le cas où le gain est le plus visible.

Viz 2 :
Groupe bas-gauche (séjours courts, erreur faible)
Fracture, Kidney Disease, Influenza, Appendicitis — durée moyenne 6-8 jours, MAE 0.68-0.80. Le modèle est précis sur ces profils.
Groupe haut-droite (séjours longs, erreur forte)
Stroke, Sepsis, Pneumonia — durée moyenne 9-11 jours, MAE 0.83-0.89. Le modèle se trompe davantage sur les cas sévères.
L'exception : Hypertension, Diabetes, et l'Appendicits sont en haut-gauche — séjours courts mais erreur élevée. Ce sont des pathologies à forte variabilité individuelle malgré une durée moyenne basse.

Ces deux viz  montrent que le modèle est moyen sur tous les diagnostics et que sa précision se dégrade sur les cas sévères Cela qui justifie précisément le passage à un modèle non linéaire capable de capturer les signaux faibles des features numériques

In [22]:
# Ajoute la colonne erreur (résidus)
results["residual"] = results["actual"] - results["predicted"]

# Visualise si l'erreur est liée à la durée réelle
fig = px.scatter(results, 
                 x="actual", 
                 y="residual", 
                 color="primary_diagnosis",
                 title="Analyse des Résidus (Erreurs)")
fig.add_hline(y=0, line_dash="dash") # La ligne de perfection
fig.show()

In [41]:
mlflow.set_experiment("length_of_stay_prediction")

with mlflow.start_run(run_name="random_forest"):

    cv_results = cross_validate(
        model_rf, X_train, y_train,
        cv=cv,
        scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"],
        return_train_score=True
    )

    cv_mae  = -cv_results["test_neg_mean_absolute_error"].mean()
    cv_rmse =  -cv_results["test_neg_root_mean_squared_error"].mean()
    cv_r2   =  cv_results["test_r2"].mean()

    print("=== Cross-validation (train set) ===")
    print(f"MAE  : {cv_mae:.3f} jours")
    print(f"RMSE : {cv_rmse:.3f} jours")
    print(f"R²   : {cv_r2:.3f}")

    model_rf.fit(X_train, y_train)
    y_pred_rf = model_rf.predict(X_test)

    test_mae  = mean_absolute_error(y_test, y_pred_rf)
    test_rmse = root_mean_squared_error(y_test, y_pred_rf)
    test_r2   = r2_score(y_test, y_pred_rf)

    print("\n=== Test set ===")
    print(f"MAE  : {test_mae:.3f} jours")
    print(f"RMSE : {test_rmse:.3f} jours")
    print(f"R²   : {test_r2:.3f}")

    mlflow.log_params({
        "model":        "RandomForest",
        "n_estimators": 100,
        "test_size":    0.2,
        "cv_folds":     5,
        "num_features": num_features,
        "cat_features": cat_features
    })

    mlflow.log_metrics({
        "cv_mae":   cv_mae,
        "cv_rmse":  cv_rmse,
        "cv_r2":    cv_r2,
        "test_mae":  test_mae,
        "test_rmse": test_rmse,
        "test_r2":   test_r2
    })

    mlflow.sklearn.log_model(model_rf, name="random_forest")


=== Cross-validation (train set) ===
MAE  : 0.851 jours
RMSE : 1.073 jours
R²   : 0.687

=== Test set ===
MAE  : 0.825 jours
RMSE : 1.051 jours
R²   : 0.717


2026/03/17 20:22:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [48]:
# 1. Créer un DataFrame de résultats sur le set de Test
results = X_test.copy()
results["actual"] = y_test
results["predicted"] = model_rf.predict(X_test)

# 3. GroupBy et calcul
r2_per_diagnosis = results.groupby("primary_diagnosis").apply(get_r2)

print(r2_per_diagnosis.sort_values())

primary_diagnosis
Appendicitis      0.051118
Sepsis            0.062466
Kidney Disease    0.091514
Influenza         0.099577
COPD              0.102957
Hypertension      0.105563
Heart Failure     0.109839
Diabetes          0.113642
Fracture          0.134234
Stroke            0.151125
Pneumonia         0.177847
dtype: float64


C:\Users\muren\AppData\Local\Temp\ipykernel_42768\525937940.py:7: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [53]:
mlflow.set_experiment("length_of_stay_prediction")

with mlflow.start_run(run_name="xgboost"):

    cv_results = cross_validate(
        model_xgb, X_train, y_train,
        cv=cv,
        scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"],
        return_train_score=True
    )

    cv_mae  = -cv_results["test_neg_mean_absolute_error"].mean()
    cv_rmse = -cv_results["test_neg_root_mean_squared_error"].mean()
    cv_r2   =  cv_results["test_r2"].mean()

    print("=== Cross-validation (train set) ===")
    print(f"MAE  : {cv_mae:.3f} jours")
    print(f"RMSE : {cv_rmse:.3f} jours")
    print(f"R²   : {cv_r2:.3f}")

    model_xgb.fit(X_train, y_train)
    y_pred_xgb = model_xgb.predict(X_test)

    test_mae  = mean_absolute_error(y_test, y_pred_xgb)
    test_rmse = root_mean_squared_error(y_test, y_pred_xgb)
    test_r2   = r2_score(y_test, y_pred_xgb)

    print("\n=== Test set ===")
    print(f"MAE  : {test_mae:.3f} jours")
    print(f"RMSE : {test_rmse:.3f} jours")
    print(f"R²   : {test_r2:.3f}")

    mlflow.log_params({
        "model":         "XGBoost",
        "n_estimators":  100,
        "learning_rate": 0.1,
        "max_depth":     4,
        "test_size":     0.2,
        "cv_folds":      5,
        "num_features":  num_features,
        "cat_features":  cat_features
    })

    mlflow.log_metrics({
        "cv_mae":    cv_mae,
        "cv_rmse":   cv_rmse,
        "cv_r2":     cv_r2,
        "test_mae":  test_mae,
        "test_rmse": test_rmse,
        "test_r2":   test_r2
    })

    mlflow.sklearn.log_model(model_xgb, name="xgboost")

    print("\n✓ Run loggé dans mlflow")

=== Cross-validation (train set) ===
MAE  : 0.784 jours
RMSE : 1.015 jours
R²   : 0.720

=== Test set ===
MAE  : 0.753 jours
RMSE : 0.987 jours
R²   : 0.750


2026/03/17 20:32:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



✓ Run loggé dans mlflow


In [56]:
# 1. Créer un DataFrame de résultats sur le set de Test
results_xgb = X_test.copy()
results_xgb["actual"] = y_test
results_xgb["predicted"] = model_xgb.predict(X_test)

# 3. GroupBy et calcul
r2_per_diagnosis_xgb = results_xgb.groupby("primary_diagnosis").apply(get_r2)

print(r2_per_diagnosis_xgb.sort_values())

primary_diagnosis
Sepsis            0.013138
Appendicitis      0.107715
Hypertension      0.206748
Pneumonia         0.208796
Stroke            0.229942
Influenza         0.241016
Fracture          0.249740
Diabetes          0.250262
Heart Failure     0.251527
COPD              0.264880
Kidney Disease    0.280801
dtype: float64


C:\Users\muren\AppData\Local\Temp\ipykernel_42768\1949415662.py:7: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [60]:
# On mappe cette moyenne sur le set de test
results_xgb["baseline_pred"] = results_xgb["primary_diagnosis"].map(baseline_means)

# Comparaison des erreurs
results_xgb["error_model_xgb"] = (results_xgb["actual"] - results_xgb["predicted"]).abs()
results_xgb["error_baseline"] = (results_xgb["actual"] - results_xgb["baseline_pred"]).abs()


df_results_xgb = results_xgb.groupby("primary_diagnosis")[["error_model_xgb","error_baseline","baseline_pred"]].mean()

print(df_results_xgb)

                   error_model_xgb  error_baseline  baseline_pred
primary_diagnosis                                                
Appendicitis              0.803389        0.816059       5.878277
COPD                      0.694559        0.846261       8.826681
Diabetes                  0.749667        0.890076       6.753741
Fracture                  0.649793        0.778724       7.813364
Heart Failure             0.666806        0.844138       9.746898
Hypertension              0.791624        0.894626       5.820907
Influenza                 0.711625        0.871858       6.793991
Kidney Disease            0.657052        0.823178       7.741935
Pneumonia                 0.766993        0.928816       8.701176
Sepsis                    0.868249        0.966783      10.758551
Stroke                    0.868038        1.022266       9.742922


In [62]:
df_results_xgb = df_results_xgb.sort_values("baseline_pred")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("MAE : modèle vs naïf par diagnostic",
                    "Durée moyenne vs erreur modèle")
)

# Viz 1 — Barres groupées
fig.add_trace(go.Bar(
    name="Modèle LR",
    x=df_results_xgb.index,
    y=df_results_xgb["error_model_xgb"],
    marker_color="#4C78A8"
), row=1, col=1)

fig.add_trace(go.Bar(
    name="Modèle naïf",
    x= df_results_xgb.index,
    y= df_results_xgb["error_baseline"],
    marker_color="#F58518"
), row=1, col=1)

# Viz 2 — Scatter durée vs erreur
fig.add_trace(go.Scatter(
    x=df_results_xgb["baseline_pred"],
    y= df_results_xgb["error_model_xgb"],
    mode="markers+text",
    text= df_results_xgb.index,
    textposition="top center",
    marker=dict(size=10, color="#4C78A8"),
    showlegend=False
), row=1, col=2)

fig.update_layout(
    barmode="group",
    template="plotly_white",
    title="Analyse des erreurs par diagnostic",
    height=500
)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_yaxes(title_text="MAE (jours)", row=1, col=1)
fig.update_xaxes(title_text="Durée moyenne séjour (jours)", row=1, col=2)
fig.update_yaxes(title_text="MAE modèle (jours)", row=1, col=2)

fig.show()

L'écart entre barre bleue (modèle xgboost) et orange (naïf) est plus large qu'avec la régression linéaire — XGBoost gagne davantage sur le modèle naïf, particulièrement sur COPD, Heart Failure et Kidney Disease. Le gain est visuellement plus marqué que sur la baseline LR.

Deux évolutions notables par rapport à la viz LR :
Le groupe bas-droite a disparu — COPD, Heart Failure et Kidney Disease qui étaient à 0.75-0.80 de MAE avec la LR descendent maintenant à 0.65-0.70. XGBoost capte mieux les interactions sur les séjours moyens-longs (8-10 jours).
La corrélation durée/erreur s'est atténuée — avec la LR la tendance montait clairement de gauche à droite. Ici le nuage est plus plat. XGBoost réduit l'hétéroscédasticité qu'on avait identifiée dans les résidus de la baseline.
Sepsis et Stroke restent isolés en haut — les deux cas les plus sévères (MAE ~0.87) ne bougent quasiment pas malgré XGBoost. C'est la confirmation que le plafond de performance sur ces diagnostics est lié à des informations absentes du dataset, pas au choix du modèle.

XGBoost améliore uniformément les cas intermédiaires mais ne résout pas les cas sévères, ce qui pointe vers un problème defeatures plutôt que de modèle.